# Arrowstack Project 1 — Customer Churn Investigation
**Alkamah Sakilur Rashid** | Data Science Internship

## Objective
Analyze a simulated subscription dataset to identify churn drivers, segment customers and propose retention-analysis actions. This notebook follows the assignment requirements: data dictionary, quality checks, business-question EDA, evidence-based insights, limitations and next steps.

> Important: the dataset is synthetic. Findings are descriptive associations inside the simulation, not causal claims about real customers.


## 1. Setup and data load


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

candidates = [Path('data/customer_churn_simulated.csv'), Path('../data/customer_churn_simulated.csv')]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not locate data/customer_churn_simulated.csv')
df = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
df.head()


In [ ]:
print('Shape:', df.shape)
display(df.describe(include='all').T)


## 2. Data dictionary
The complete field-level dictionary is maintained in `docs/data_dictionary.md`. Key analytical fields are contract type, satisfaction, support tickets, late payments, tenure, autopay and churn.


## 3. Data quality and cleaning checks
Business rule: do not silently repair suspicious values. First quantify missingness, duplicates and domain violations. The committed dataset is already clean; the notebook verifies that claim.


In [ ]:
required = ['customer_id','tenure_months','plan_type','monthly_charges','total_charges','contract_type','payment_method','support_tickets','tech_issues','usage_hours_month','late_payments','satisfaction_score','autopay','region','churn']
assert set(required).issubset(df.columns)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate customer IDs:', int(df['customer_id'].duplicated().sum()))
print('Invalid satisfaction:', int(((df['satisfaction_score'] < 1) | (df['satisfaction_score'] > 5)).sum()))
print('Invalid churn labels:', int((~df['churn'].isin([0,1])).sum()))
assert df['customer_id'].is_unique
assert df.isna().sum().sum() == 0
assert df['satisfaction_score'].between(1,5).all()
assert df['churn'].isin([0,1]).all()
print('QUALITY CHECK: PASS')


## 4. Business Question 1 — What is the baseline churn rate?


In [ ]:
overall = df['churn'].mean()
print(f'Overall churn rate: {overall:.1%}')
display(df['churn'].value_counts().rename({0:'Retained',1:'Churned'}).to_frame('customers'))


## 5. Business Question 2 — Does contract type show a different churn pattern?


In [ ]:
contract_summary = (df.groupby('contract_type')['churn']
                    .agg(customers='size', churn_rate='mean')
                    .sort_values('churn_rate', ascending=False))
contract_summary['churn_rate'] = contract_summary['churn_rate'] * 100
display(contract_summary)

plt.figure(figsize=(8,5))
sns.barplot(data=contract_summary.reset_index(), x='contract_type', y='churn_rate')
plt.ylabel('Churn rate (%)'); plt.xlabel('Contract type'); plt.title('Churn rate by contract type'); plt.xticks(rotation=15); plt.tight_layout(); plt.show()


**Observed pattern:** month-to-month customers have a 74.6% simulated churn rate versus 34.1% for one-year and 25.7% for two-year contracts. This is an association in the synthetic dataset, not evidence that contract type itself causes churn.


## 6. Business Question 3 — How does satisfaction relate to churn?


In [ ]:
df['satisfaction_band'] = pd.cut(df['satisfaction_score'], bins=[0,2,3,5], labels=['1-2','3','4-5'])
sat_summary = df.groupby('satisfaction_band', observed=True)['churn'].agg(customers='size', churn_rate='mean')
sat_summary['churn_rate'] *= 100
display(sat_summary)


**Observed pattern:** churn is 68.8% for satisfaction 1–2, 51% for satisfaction 3, and 32.8% for satisfaction 4–5. Lower satisfaction bands are associated with higher simulated churn.


## 7. Business Question 4 — Is support burden concentrated among churned customers?


In [ ]:
support_summary = (df.assign(support_band=np.where(df['support_tickets']>=4,'4+',df['support_tickets'].astype(str)))
                  .groupby('support_band', sort=False)['churn']
                  .agg(customers='size', churn_rate='mean'))
support_summary['churn_rate'] *= 100
display(support_summary)

plt.figure(figsize=(8,5))
sns.barplot(data=support_summary.reset_index(), x='support_band', y='churn_rate')
plt.ylabel('Churn rate (%)'); plt.xlabel('Support-ticket band'); plt.title('Churn rate by support burden'); plt.tight_layout(); plt.show()


Interpret support volume as a diagnostic signal. A support ticket can be a symptom of product friction rather than a cause of churn, so the next step should be qualitative issue analysis and time-aware validation.


## 8. Business Question 5 — What does payment behavior show?


In [ ]:
autopay_summary = df.groupby('autopay')['churn'].agg(customers='size', churn_rate='mean')
autopay_summary['churn_rate'] *= 100
display(autopay_summary)


Autopay groups differ descriptively in this simulation: 39.6% churn for Yes versus 49.9% for No. This should be treated as a segment signal, not a causal claim.


## 9. Investigation segment
To turn the EDA into an actionable investigation queue, define a transparent descriptive segment: month-to-month contract + at least 3 support tickets + satisfaction score <= 3.


In [ ]:
risk_segment = df[(df['contract_type']=='Month-to-month') & (df['support_tickets']>=3) & (df['satisfaction_score']<=3)].copy()
print('Segment size:', len(risk_segment), f'({len(risk_segment)/len(df):.1%} of dataset)')
print('Segment churn rate:', risk_segment['churn'].mean())
display(risk_segment[['customer_id','contract_type','support_tickets','satisfaction_score','late_payments','churn']].head(10))


The segment contains 115 records (9.6% of the dataset). It is a prioritization lens for further analysis, not an automated customer treatment rule.


## 10. Evidence-based findings
1. Contract commitment is strongly separated in this simulation: month-to-month churn is materially higher than the two longer contract groups.
2. Lower satisfaction bands have higher churn rates.
3. Support burden is useful as a diagnostic segmentation dimension and should be investigated with issue categories and timing.
4. Payment behavior differs across autopay groups, but the analysis does not establish why.
5. A transparent multi-signal segment can focus retention investigation without pretending to predict individual outcomes.


## 11. Limitations and next steps
- Synthetic data cannot establish real-world prevalence or effect size.
- Relationships are partly encoded by the simulation design.
- No causal inference, intervention test or external benchmark is available.
- Next: validate on authorized production data, add cohort/time analysis, test retention interventions with measurable outcomes, monitor schema/churn drift, and review privacy/fairness before operational use.


## 12. Final QA checklist
- [x] Data dictionary
- [x] Data cleaning and quality checks
- [x] Business-question EDA
- [x] Evidence-based insights
- [x] Limitations and next steps
- [x] Reproducible artifact
- [x] Validation evidence
- [x] Professional handoff package
